# Домашнее задание 2. Своя среда: управление запасами

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/02-environments/homework/homework.ipynb)

**Вес в оценке:** базовое ДЗ, среднее по сложности.

Задание состоит из трёх частей и бонуса:

1. **Практика: среда `InventoryEnv` как `gym.Env` и ручные политики (30 баллов)**
2. **Практика: ценность и уравнения Беллмана на складе (40 баллов)**
3. **Теория: уравнения Беллмана и MDP на бумаге (30 баллов)**
4. **Бонус: скрытый день недели — наблюдение, которое чинит марковость (10 баллов)**

Части с `assert` проверяются автоматически при запуске ячейки — если assert не упал, эта часть засчитана. Текстовые ответы и графики проверяются вручную.

## Задача

Вы управляете складом одного товара. Каждый день:

1. утром на складе `stock` единиц (от 0 до `capacity`);
2. вы решаете, сколько единиц **заказать**: действие $a \in \{0, 1, \ldots, \text{max\_order}\}$; заказ приезжает сразу, но склад не может вместить больше `capacity` — лишнее пропадает;
3. днём приходит **случайный спрос** $d$ (например, Пуассон со средним `demand_mean`), продаётся $\min(d, \text{stock})$ единиц;
4. вечером остаток переходит на завтра. Эпизод длится `horizon` дней.

Экономика дня: выручка `price` за проданную единицу, `order_cost` за заказанную, `holding_cost` за каждую единицу, оставшуюся на складе к вечеру, и `stockout_cost` за каждую единицу неудовлетворённого спроса.

Это классическая задача исследования операций, и RL здесь — не единственный способ, зато прекрасный полигон: дискретное состояние, дискретное действие, честная случайность, и — главное для этой недели — распределение спроса известно, а значит, матрицы $P$ и $R$ можно выписать **точно** и решить уравнения Беллмана, как на лекции.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

## Часть 1. Среда `InventoryEnv` и ручные политики (30 баллов)

### 1.1 Класс среды (20 баллов)

Реализуйте `InventoryEnv`:

* `observation_space = Discrete(capacity + 1)` — наблюдение = текущий остаток;
* `action_space = Discrete(max_order + 1)`;
* `reset(seed)` — остаток `initial_stock`, день 0, вернуть `(obs, info)`;
* `step(action)` — по описанию выше; награда за день = выручка − стоимость заказа − хранение − штраф за дефицит; `terminated=True` после `horizon`-го дня; в `info` положите `demand` и `sold`;
* спрос генерируйте через `self.np_random.poisson(self.demand_mean)` — тогда среда воспроизводима по сиду.

Подумайте, какой из флагов — `terminated` или `truncated` — правильно выставлять в конце горизонта, и объясните выбор в комментарии. (Подсказка: после `horizon` дней задача **закончена по условию**, а не оборвана снаружи.)

In [ ]:
class InventoryEnv(gym.Env):
    metadata = {"render_modes": ["ansi"]}

    def __init__(self, capacity=20, max_order=10, demand_mean=4.0, horizon=30, initial_stock=10,
                 price=5.0, order_cost=2.0, holding_cost=0.5, stockout_cost=3.0, render_mode=None):
        self.capacity, self.max_order = capacity, max_order
        self.demand_mean, self.horizon, self.initial_stock = demand_mean, horizon, initial_stock
        self.price, self.order_cost = price, order_cost
        self.holding_cost, self.stockout_cost = holding_cost, stockout_cost
        self.render_mode = render_mode

        # TODO: observation_space и action_space
        self.observation_space = None
        self.action_space = None

        self.stock = initial_stock
        self.day = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # TODO
        raise NotImplementedError

    def step(self, action):
        # TODO: заказ -> спрос -> продажа -> награда -> конец горизонта
        raise NotImplementedError

    def render(self):
        if self.render_mode == "ansi":
            return f"день {self.day:2d}: на складе {self.stock:2d}"

In [ ]:
# Самопроверка 1.1
env = InventoryEnv()
check_env(env)

obs, info = env.reset(seed=0)
assert obs == 10 and env.observation_space.contains(obs)
assert env.action_space.n == 11 and env.observation_space.n == 21

# заказ выше вместимости обрезается: 10 + 10 = 20 -> не больше capacity
env.reset(seed=0)
obs, r, term, trunc, info = env.step(10)
assert 0 <= obs <= 20 and "demand" in info and "sold" in info
assert info["sold"] <= 20 and obs == 20 - info["sold"], "продано не может быть больше остатка после заказа"

# эпизод длится ровно horizon дней
env.reset(seed=1)
for day in range(30):
    obs, r, term, trunc, info = env.step(0)
    assert term == (day == 29), "terminated должен стать True ровно после horizon-го дня"

# воспроизводимость: одинаковый seed -> одинаковый спрос
def demands(seed):
    e = InventoryEnv(); e.reset(seed=seed)
    return [e.step(3)[4]["demand"] for _ in range(10)]
assert demands(42) == demands(42) and demands(42) != demands(43)
print("OK: InventoryEnv проходит проверки")

### 1.2 Базовые линии (10 баллов)

Реализуйте функцию `evaluate(env_factory, policy, n_episodes, seed)` — средний return политики `policy(obs) -> action` по `n_episodes` эпизодам — и сравните три ручные политики:

1. **ничего не заказывать**;
2. **заказывать фиксированно** `k` единиц каждый день (подберите `k` перебором от 0 до 10);
3. **пополнять до уровня** $S$: заказать $\max(0, S - \text{stock})$ (классическая «base-stock policy»; подберите $S$).

Постройте график «средний return в зависимости от параметра» для политик 2 и 3. Какая из ручных политик лучшая и почему это ожидаемо?

In [ ]:
def evaluate(env_factory, policy, n_episodes=100, seed=0):
    # TODO: средний return по n_episodes эпизодам; сиды эпизодов seed, seed+1, ...
    raise NotImplementedError

# TODO: три политики, перебор параметров, график

In [ ]:
# Самопроверка 1.2
_r = evaluate(InventoryEnv, lambda obs: 0, n_episodes=20)
assert isinstance(_r, float) or isinstance(_r, np.floating)
print(f"OK: evaluate работает; return политики «ничего не заказывать» = {_r:.1f}")

## Часть 2. Ценность и уравнения Беллмана на складе (40 баллов)

### 2.1 Точная модель MDP (15 баллов)

Спрос — Пуассон с известным средним, значит, ядро переходов можно выписать явно. Реализуйте `inventory_mdp(env)`, возвращающую `P[s, a, s']` и `R[s, a]` для `InventoryEnv` (состояние — остаток $s \in \{0, \ldots, \text{capacity}\}$, действие — заказ $a$):

* после заказа на складе $m = \min(s + a, \text{capacity})$ единиц;
* при спросе $d$ продаётся $\min(d, m)$, остаток $s' = \max(m - d, 0)$; все значения спроса $d \ge m$ приводят в $s' = 0$ — не забудьте просуммировать **хвост** распределения (функция `poisson_pmf` ниже уже это делает);
* $R[s, a]$ — **ожидаемая** награда дня: $\mathbb E_d[\,\text{price}\cdot\min(d, m) - \text{order\_cost}\cdot a - \text{holding\_cost}\cdot\max(m - d, 0) - \text{stockout\_cost}\cdot\max(d - m, 0)\,]$; хвост распределения можно обрезать на $d_{\max} = 60$ (остаток вероятности пренебрежимо мал);
* горизонт в этой части считаем бесконечным, а задачу — дисконтированной с $\gamma = 0.95$: тогда ценность не зависит от дня, и уравнения Беллмана из лекции применимы дословно. (Как учесть конечный горизонт — вопрос 3.4.)

In [ ]:
def poisson_pmf(mean, d_max=60):
    """Вероятности спроса d = 0..d_max для Пуассона; хвост d > d_max добавлен к последней точке."""
    pmf = np.zeros(d_max + 1)
    pmf[0] = np.exp(-mean)
    for d in range(1, d_max + 1):
        pmf[d] = pmf[d - 1] * mean / d
    pmf[-1] += 1 - pmf.sum()
    return pmf

def inventory_mdp(env, d_max=60):
    """P[s, a, s'] и R[s, a] для InventoryEnv при бесконечном горизонте."""
    n_s, n_a = env.capacity + 1, env.max_order + 1
    P, R = np.zeros((n_s, n_a, n_s)), np.zeros((n_s, n_a))
    pmf = poisson_pmf(env.demand_mean, d_max)
    # TODO: для каждого s, a: m = min(s + a, capacity); для каждого d: s' = max(m - d, 0), награда дня,
    #       P[s, a, s'] += pmf[d], R[s, a] += pmf[d] * награда
    raise NotImplementedError

GAMMA = 0.95
env = InventoryEnv()
P, R = inventory_mdp(env)

# Самопроверка 2.1
assert P.shape == (21, 11, 21) and R.shape == (21, 11)
assert np.allclose(P.sum(axis=2), 1.0), "строки P — распределения"
assert np.all(P[:, :, 0] > 0), "нулевой остаток достижим из любого состояния (большой спрос)"
assert abs(P[20, 5, 20] - np.exp(-env.demand_mean)) < 1e-12, "полный склад остаётся полным только при нулевом спросе"
# ожидаемая награда при s=10, a=0 сходится с Монте-Карло по среде
_e = InventoryEnv(); _rs = []
for _seed in range(3000):
    _e.reset(seed=_seed); _rs.append(_e.step(0)[1])
assert abs(np.mean(_rs) - R[10, 0]) < 0.3, f"R[10, 0] = {R[10, 0]:.2f}, а Монте-Карло даёт {np.mean(_rs):.2f}"
print("OK: модель MDP склада построена")

### 2.2 Ценность ручной политики: точно и по Монте-Карло (10 баллов)

Возьмите лучшую base-stock политику из части 1.2 и запишите её таблицей $\pi[s, a]$. Посчитайте $V^\pi$ двумя способами: точно, как решение $(I - \gamma P_\pi)^{-1} r_\pi$, и по Монте-Карло — средний дисконтированный return по 2000 эпизодам из каждого стартового остатка (для этого сделайте `InventoryEnv(initial_stock=s, horizon=200)`: длинный горизонт имитирует бесконечный). Постройте график «остаток → $V^\pi$» с обеими кривыми и объясните, почему они должны совпасть и где расходятся.

In [ ]:
def policy_evaluation_exact(P, R, policy, gamma=GAMMA):
    # TODO: V^π = (I - γ P_π)^{-1} r_π
    raise NotImplementedError

# TODO: таблица base-stock политики, точная V^π, Монте-Карло V^π по стартовым остаткам, график

# Самопроверка 2.2 (подставьте свою политику в pi_base)
V_base = policy_evaluation_exact(P, R, pi_base)
assert V_base.shape == (21,) and np.all(np.diff(V_base) > -1e-9), "ценность не убывает с ростом остатка: товар на складе не бывает вреден при разумной политике"
print(f"OK: V^π(остаток 10) = {V_base[10]:.1f}")

### 2.3 Оптимальная политика заказов (15 баллов)

Реализуйте value iteration и найдите $V^*$ и $Q^*$ для склада. Постройте оптимальную политику $\pi^*(s) = \arg\max_a Q^*(s, a)$ и изобразите её графиком «остаток → заказ». Ответьте текстом:

* похожа ли $\pi^*$ на «пополнять до уровня $S$»? Совпала ли она с лучшей ручной политикой из 1.2? Если да — это не случайность: для такой модели склада оптимальность политики вида base-stock — классический результат теории управления запасами, и уравнение оптимальности его «переоткрыло»; объясните своими словами, почему заказывать до фиксированного уровня разумно;
* сравните $V^*(10)$ и $V^\pi(10)$ ручной политики; во сколько эпизодов обошлось value iteration и во сколько — Cross-Entropy из недели 1 (код дан ниже, обучите его и сравните честной оценкой на среде с `horizon=200`). Почему Cross-Entropy с таблицей $21 \times 11$ проигрывает, хотя у него было несколько тысяч эпизодов?

In [ ]:
def value_iteration(P, R, gamma=GAMMA, tol=1e-8):
    # TODO: V ← max_a [R[s, a] + γ Σ_s' P[s, a, s'] V[s']] до сходимости; вернуть V*, Q*
    raise NotImplementedError

def run_session(env, policy, rng, max_steps=1000):
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, total = [], [], 0.0
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[obs]))
        states.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return states, actions, total

def cross_entropy_method(env, n_iter=30, n_sessions=150, q=0.6, laplace=1.0, mix=0.5, seed=0):
    """Табличный Cross-Entropy из недели 1 (по недисконтированной сумме за эпизод)."""
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions
    for _ in range(n_iter):
        sessions = [run_session(env, policy, rng) for _ in range(n_sessions)]
        returns = np.array([G for *_, G in sessions])
        threshold = np.quantile(returns, q)
        counts = np.full((n_states, n_actions), laplace)
        for states, actions, G in sessions:
            if G >= threshold and G > returns.min():
                for s, a in zip(states, actions):
                    counts[s, a] += 1
        policy = mix * counts / counts.sum(axis=1, keepdims=True) + (1 - mix) * policy
    return policy

# TODO: V*, Q*, π*, график «остаток → заказ», сравнение с ручной политикой и Cross-Entropy, текстовые ответы

# Самопроверка 2.3
V_star, Q_star = value_iteration(P, R)
pi_star = np.eye(11)[Q_star.argmax(axis=1)]
assert np.allclose(V_star, Q_star.max(axis=1))
assert np.all(V_star >= V_base - 1e-6), "V* не меньше ценности ручной политики во всех состояниях"
assert np.allclose(policy_evaluation_exact(P, R, pi_star), V_star, atol=1e-4), "жадная по Q* политика имеет ценность V*"
print(f"OK: V*(10) = {V_star[10]:.1f}, у ручной политики V^π(10) = {V_base[10]:.1f}")

## Часть 3. Теория: уравнения Беллмана и MDP на бумаге (30 баллов)

Ответы пишите в ячейке ниже (Markdown, можно с формулами).

### 3.1 Вывод уравнения Беллмана (8 баллов)

Исходя из определения $V^\pi(s) = \mathbb E_\pi[G_t \mid S_t = s]$ и тождества $G_t = R_t + \gamma G_{t+1}$, выведите уравнение Беллмана для $V^\pi$. Явно укажите, где используется марковское свойство и где — то, что политика не зависит от времени.

### 3.2 Связь $V$ и $Q$ (5 баллов)

Докажите, что $V^\pi(s) = \sum_a \pi(a \mid s) Q^\pi(s, a)$ и $Q^\pi(s, a) = r(s, a) + \gamma \sum_{s'} P(s' \mid s, a) V^\pi(s')$. Выведите из них, что $V^*(s) = \max_a Q^*(s, a)$, и объясните, почему $\max$ появляется именно здесь, а не в определении $Q^*$.

### 3.3 Ценность руками (5 баллов)

Цепочка из трёх состояний $s_0 \to s_1 \to s_2$, политика детерминированная (всегда «вправо»), переходы детерминированные, награда $+1$ за приход в $s_2$, $s_2$ терминальное, $\gamma = 0.9$. Выпишите уравнения Беллмана для $V^\pi(s_0)$ и $V^\pi(s_1)$ и решите их. Затем сделайте переход $s_1 \to s_2$ случайным: с вероятностью $0.5$ агент остаётся в $s_1$. Решите заново и объясните, почему $V^\pi(s_1)$ стало меньше единицы.

### 3.4 Склад как MDP (6 баллов)

Опишите `InventoryEnv` как MDP $\langle \mathcal S, \mathcal A, P, R, \gamma \rangle$: выпишите $\mathcal S$, $\mathcal A$, формулу для $P(s' \mid s, a)$ через распределение спроса (с обрезанием по `capacity` и суммированием хвоста в $s' = 0$) и $r(s, a)$. В среде эпизод длится `horizon` дней и заканчивается с `terminated=True`. Объясните, что это означает для уравнения Беллмана: почему ценность последнего дня равна ожидаемой награде за день, и как выглядит уравнение для $V_t(s)$ при конечном горизонте (ценность зависит от номера дня $t$). Как бы вы включили номер дня в состояние, чтобы вернуться к стационарной постановке из лекции?


### 3.5 Лотерея Колобка (6 баллов)

Колобок стоит у киоска (состояние $s$). Действие «купить билет» стоит $c = 10$; с вероятностью $p$ билет проигрышный, и Колобок снова в $s$, с вероятностью $1 - p$ — выигрыш $W = 1000$ и конец эпизода. Действие «не покупать» — награда $0$ и конец эпизода. Возьмите $\gamma = 1$.

1. Выпишите уравнение оптимальности Беллмана для $Q^*(s, \text{купить})$ (оно есть в разделе 5 лекции) и решите его: при каких $p$ покупать выгодно, при каких — нет, и чему равно $Q^*(s, \text{купить})$ в каждом случае.
2. Найдите порог $p^*$, при котором «купить» и «не покупать» равноценны, для произвольных $c$ и $W$. Проверьте, что при $c = 10$, $W = 1000$ получается $p^* = 0.99$.
3. Теперь $\gamma < 1$. Как меняется порог $p^*$ и почему?

*Ваши ответы на часть 3:*

**3.1**

**3.2**

**3.3**

**3.4**

**3.5**

## Бонус. Скрытый день недели (10 баллов)

Иллюстрация марковского свойства из раздела 6 лекции. Пусть спрос зависит от дня недели: в будни средний спрос `demand_mean`, в субботу и воскресенье — в `weekend_factor` раз больше (по умолчанию втрое: в выходные спрос выше, чем можно заказать за один день, и выгодно запасаться заранее). День недели — это `day % 7` (день 0 — понедельник).

Реализуйте `WeekdayDemandEnv(InventoryEnv)` с параметром `observe_weekday`:

* `observe_weekday=False`: наблюдение — только остаток, как раньше (`Discrete(capacity + 1)`). Агент не видит, что завтра выходные, — марковское свойство нарушено *нашим выбором наблюдения*;
* `observe_weekday=True`: наблюдение — пара (остаток, день недели), закодированная одним числом `stock * 7 + weekday` в `Discrete((capacity + 1) * 7)`.

Сначала сравните **ручные** политики: обычную «пополнять до уровня $S$» и её версию с двумя уровнями — $S$ в будни и $S_w$ перед выходными (в пятницу и субботу). Затем обучите Cross-Entropy в обоих вариантах наблюдения (одинаковые гиперпараметры, 30 итераций, несколько сидов) и оцените всё **одной и той же** честной функцией `evaluate` на среде с `observe_weekday=True` (политике без дня недели передавайте `obs // 7`). Ответьте текстом: почему вариант с днём недели выигрывает, во сколько раз выросла таблица политики и как это сказалось на скорости обучения. Что будет, если спрос зависит от дня недели, а `weekend_factor = 1.0`?

In [ ]:
class WeekdayDemandEnv(InventoryEnv):
    def __init__(self, observe_weekday=False, weekend_factor=3.0, **kwargs):
        super().__init__(**kwargs)
        self.observe_weekday, self.weekend_factor = observe_weekday, weekend_factor
        # TODO: observation_space в зависимости от observe_weekday

    def _obs(self):
        # TODO: остаток или stock * 7 + weekday
        raise NotImplementedError

    def _demand_mean_today(self):
        # TODO: demand_mean * weekend_factor в субботу и воскресенье (self.day % 7 in (5, 6)), иначе demand_mean
        raise NotImplementedError

    def reset(self, seed=None, options=None):
        # TODO: как в InventoryEnv, но вернуть self._obs()
        raise NotImplementedError

    def step(self, action):
        # TODO: как в InventoryEnv, но спрос ~ Poisson(self._demand_mean_today()) и наблюдение self._obs()
        raise NotImplementedError

# TODO: обучение Cross-Entropy в двух вариантах, честная оценка, сравнение, текстовый ответ

In [ ]:
# Самопроверка бонуса
for flag, n_obs in [(False, 21), (True, 21 * 7)]:
    _e = WeekdayDemandEnv(observe_weekday=flag)
    check_env(_e)
    assert _e.observation_space.n == n_obs, f"observe_weekday={flag}: ожидали Discrete({n_obs})"
_e = WeekdayDemandEnv(observe_weekday=True)
obs, _ = _e.reset(seed=0)
assert obs == 10 * 7 + 0, "в день 0 (понедельник) при остатке 10 наблюдение должно быть 70"
# в выходные спрос в среднем выше: усредняем по многим сидам
def mean_demand(day_index, n=300):
    total = 0.0
    for s in range(n):
        e = WeekdayDemandEnv(); e.reset(seed=s)
        for d in range(day_index + 1):
            info = e.step(10)[4]
        total += info["demand"]
    return total / n
assert mean_demand(5) > 1.5 * mean_demand(2), "в субботу спрос должен быть заметно выше, чем в среду"
print("OK: WeekdayDemandEnv проходит проверки")

## Чек-лист перед сдачей

- [ ] `InventoryEnv` проходит `check_env` и самопроверку 1.1; ручные политики оценены, есть график по параметру
- [ ] Модель MDP склада проходит самопроверку 2.1
- [ ] Ценность ручной политики посчитана точно и по Монте-Карло, есть график и объяснение
- [ ] Value iteration реализован, есть график оптимальной политики заказов и сравнение с ручной политикой и Cross-Entropy
- [ ] Часть 3: вывод уравнения Беллмана, связь $V$–$Q$, расчёт руками, склад как MDP, лотерея Колобка
- [ ] (бонус) `WeekdayDemandEnv` проходит самопроверку, есть сравнение двух наблюдений и объяснение